# FAMILY HEALTH Knowledge Worker
- A conversational AI about the Miller/Vance family. 
- Ask questions about the family relationships, each member's health history including medication, immunizations, imaging, and more. 

In [ ]:
# imports

import os
import glob
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv
from chromadb import PersistentClient

In [ ]:
load_dotenv(override=True)
client = OpenAI()

LLM = 'gpt-4o-mini'
EMBEDDING_MODEL = 'text-embedding-3-small'
KB_PATH = 'knowledge-base'
DB_NAME = 'chroma_db'
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

In [ ]:
# load documents

def load_documents():
    docs = []
    for path in glob.glob(f'{KB_PATH}/**/*.md', recursive=True):
        with open(path, 'r', encoding='utf-8') as f:
            docs.append({'source': path, 'text': f.read()})
    print(f'Loaded {len(docs)} documents')
    return docs

docs = load_documents()

In [ ]:
# create chunks

def chunk_text(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start:start + size])
        start += size - overlap
    return chunks

def build_chunks(docs):
    all_chunks = []
    for doc in docs:
        for chunk in chunk_text(doc['text']):
            all_chunks.append({'text': chunk, 'source': doc['source']})
    print(f'Created {len(all_chunks)} chunks')
    return all_chunks

chunks = build_chunks(docs)

In [ ]:
# embed and store in chroma 

def build_vectorstore(chunks):
    texts = [c['text'] for c in chunks]
    metas = [{'source': c['source']} for c in chunks]

    embeddings = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    vectors = [e.embedding for e in embeddings.data]

    db = PersistentClient(path=DB_NAME)
    if 'knowledge' in [c.name for c in db.list_collections()]:
        db.delete_collection('knowledge')
    collection = db.get_or_create_collection('knowledge')

    ids = [str(i) for i in range(len(chunks))]
    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f'Vector store ready with {collection.count()} chunks')
    return collection

collection = build_vectorstore(chunks)

In [ ]:
# RAG chat

SYSTEM = """You are a private health assistant with access to the Family Health Knowledge Base.
Answer questions accurately using only the context provided from the knowledge base.
If the answer is not in the context, say you don't know.
"""

def retrieve(question, k=5):
    query_vec = client.embeddings.create(model=EMBEDDING_MODEL, input=[question]).data[0].embedding
    results = collection.query(query_embeddings=[query_vec], n_results=k)
    return '\n\n'.join(results['documents'][0])

def chat(message, history):
    context = retrieve(message)
    messages = [
        {"role": "system", "content": SYSTEM + f"\n\nContext:\n{context}"}
    ] + history

    response = client.chat.completions.create(model=LLM, messages=messages, stream=True)

    history = history + [{"role": "assistant", "content": ""}]
    result = ""
    for chunk in response:
        result += chunk.choices[0].delta.content or ""
        history[-1]["content"] = result
        # print(history)  # temporary debug
        yield history

In [ ]:
# Create Gradio User Interface

with gr.Blocks() as ui:
    gr.Markdown("## Miller/Vance Family Knowledge Worker")
    gr.Markdown(
        """
        Welcome to the Miller/Vance Family Assistant! A conversational AI grounded in a Miller/Vance Family knowledge base. Ask me anything about the family relationship, health history, including medication, immunizations, imaging,and more!
        """
        )
    chatbot = gr.Chatbot(height=500, type="messages")
    with gr.Row():
        question_box = gr.Textbox(
            lines=2, scale=4,
            placeholder="e.g. Who was diagnose with Breast Cancer?",
            label="Type in your question"
        )
        submit_btn = gr.Button("Ask", variant="primary", scale=1)

    def handle(question, history):
        if history is None:
            history = []
        history.append({"role": "user", "content": question})    
        return history 

    submit_btn.click(
        fn=lambda: gr.update(interactive=False),
        outputs=submit_btn
    ).then(
        fn=handle,
        inputs=[question_box, chatbot],
        outputs=chatbot
    ).then(
        fn=chat,
        inputs=[question_box, chatbot],
        outputs=chatbot
    ).then(
        fn=lambda: ("", gr.update(interactive=True)),
        outputs=[question_box, submit_btn]
    )

ui.launch(inbrowser=True)